# Physics-informed neural networks for solving forward and inverse flow problems via the Boltzmann-BGK formulation

**Paper:** Lou, Q., Meng, X., Karniadakis, G.E. (2021). *Physics-informed neural networks for solving forward and inverse flow problems via the Boltzmann-BGK formulation.* Journal of Computational Physics, 447, 110676.

**Carpeta origen:** `PINNs/2. Termidinamica y cinetica/Physics-informed-neural-networks-for-solving-forward-an_2021_Journal-of-Comp.pdf`

## Como se usan las PINNs en este paper

El paper propone **PINN-BGK**: una PINN para resolver la ecuacion de Boltzmann con el modelo de colision BGK discretizada en velocidades (DVB, Eq. 5):

$$\partial_t f_i + \boldsymbol\xi_i\cdot\nabla f_i = -\frac{1}{\tau}(f_i-f_i^{eq}),\quad i=0,\dots,Q-1$$

La arquitectura (Fig. 1) usa **dos subredes**: $\mathcal{NN}_{eq}(\mathbf{x},t)\to(\rho,\mathbf{u})$ predice las variables macroscopicas, de las que se obtiene $f_i^{eq}$ via la expansion de Hermite de segundo orden (Eq. 6); y $\mathcal{NN}_{neq}(\mathbf{x},t)\to f_i^{neq}$ predice directamente la parte de no-equilibrio. La distribucion completa es $f_i=f_i^{eq}+f_i^{neq}$. Esta descomposicion en dos redes (en vez de una sola) es clave: como $\tau$ es pequeno para flujos continuos, $f_i^{neq}$ es tipicamente $10^4$ veces mas pequeno que $f_i^{eq}$, y una unica red no logra resolver ambas escalas simultaneamente (Fig. 2, "Case A vs Case B" del paper) &mdash; por eso se **reescala** la salida de $\mathcal{NN}_{neq}$ por un factor (aqui $10^4$) para que la red trabaje en un rango numerico $O(1)$.

La perdida total combina el residuo de la ecuacion (Eq. 14-15) y las condiciones de contorno/iniciales:

$$L=L_{Eq}+L_{IC}+L_{BC},\qquad R_i:=\partial_t f_i+\boldsymbol\xi_i\cdot\nabla f_i+\frac{1}{\tau}(f_i-f_i^{eq})$$

Este cuaderno reproduce fielmente el **benchmark de flujo de Kovasznay** (Seccion 3.1, forward, estacionario), usando el modelo de velocidades discretas D2Q9 (Eq. 23), con la solucion analitica exacta (Eq. 21) para generar las condiciones de contorno de Dirichlet.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio, ni se encontro uno especifico al buscar en GitHub. Como referencia general del framework PINN base sobre el que se construye PINN-BGK:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Flujo de Kovasznay: solucion exacta (Eq. 21-22) y modelo de velocidades discretas D2Q9 (Eq. 23)

In [ ]:
Re, u0, p0, RT, Cconst, L = 10.0, 0.1581, 0.05, 100.0, 100.0, 1.0
nu = L * u0 / Re
tau_relax = nu / RT
lam = Re / 2 - np.sqrt(Re**2 / 4 + 4 * np.pi**2)
print(f'lambda={lam:.4f}, nu={nu:.5f}, tau={tau_relax:.3e}')

def exact_uvp(x, y):
    u = u0 * (1 - np.exp(lam * x) * np.cos(2 * np.pi * y))
    v = u0 * (lam / (2 * np.pi)) * np.exp(lam * x) * np.sin(2 * np.pi * y)
    p = p0 * (1 - np.exp(2 * lam * x)) + Cconst
    return u, v, p

# Modelo D2Q9 (Eq. 23)
c_lat = np.sqrt(3 * RT)
e = np.array([[0, 0], [1, 0], [0, 1], [-1, 0], [0, -1], [1, 1], [-1, 1], [-1, -1], [1, -1]], dtype=float)
xi = c_lat * e  # velocidades discretas xi_i = c*e_i
w = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])
Q = 9

xi_t = torch.tensor(xi, dtype=torch.float32, device=device)
w_t = torch.tensor(w, dtype=torch.float32, device=device)

## 2. Dos subredes: $\mathcal{NN}_{eq}$ (macroscopicas) y $\mathcal{NN}_{neq}$ (no-equilibrio, reescalada), Fig. 1

In [ ]:
NEQ_SCALE = 1.0e4  # f_neq ~ 10^4 veces menor que f_eq para flujos continuos (Seccion 2.2.1 y Fig. 2)

class SubNet(nn.Module):
    def __init__(self, n_out, n_hidden=4, n_neurons=40):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, n_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)


NN_eq = SubNet(3).to(device)   # -> rho, u, v
NN_neq = SubNet(Q).to(device)  # -> f_i^neq * NEQ_SCALE, para cada una de las 9 velocidades


def f_equilibrium(rho, u, v):
    """Eq. (6): expansion de Hermite de 2do orden de la Maxwelliana.
    rho, u, v: (N, 1). Devuelve (N, Q)."""
    xi_dot_u = u * xi_t[:, 0] + v * xi_t[:, 1]                 # (N,1)*(Q,) -> (N, Q)
    u_dot_u = u**2 + v**2                                       # (N, 1), broadcastea sobre Q
    return w_t * rho * (1 + xi_dot_u / RT + xi_dot_u**2 / (2 * RT**2) - u_dot_u / (2 * RT))


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]

## 3. Residuo del DVB (Eq. 15, sin termino temporal para el caso estacionario) y perdida

In [ ]:
def get_fields(xy):
    out_eq = NN_eq(xy)
    rho, u, v = out_eq[:, 0:1], out_eq[:, 1:2], out_eq[:, 2:3]
    f_eq = f_equilibrium(rho, u, v)                              # (N, Q)
    f_neq = NN_neq(xy) / NEQ_SCALE                               # (N, Q)
    f = f_eq + f_neq
    return rho, u, v, f_eq, f_neq, f


def dvb_residual(xy):
    rho, u, v, f_eq, f_neq, f = get_fields(xy)
    residuals = []
    for i in range(Q):
        fi = f[:, i:i + 1]
        fi_x = d_d(fi, xy, 0)
        fi_y = d_d(fi, xy, 1)
        Ri = xi_t[i, 0] * fi_x + xi_t[i, 1] * fi_y + (1.0 / tau_relax) * (fi - f_eq[:, i:i + 1])
        residuals.append(Ri)
    return torch.cat(residuals, dim=1), f_neq


def moment_constraint(f_neq):
    """Por construccion del modelo (Eq. 7), rho y rho*u son enteramente momentos de f_eq;
    f_neq no debe aportar masa ni momento (solo captura los momentos de orden superior /
    esfuerzos). Sin esta restriccion, NN_neq puede absorber cualquier discrepancia de
    (rho,u,v) sin que el residuo puntual R_i lo penalice, dejando (rho,u,v) subdeterminadas
    en el interior. Se impone Sum_i f_neq_i = 0 y Sum_i xi_i f_neq_i = 0."""
    mass_neq = torch.sum(f_neq, dim=1, keepdim=True)
    mom_x_neq = torch.sum(f_neq * xi_t[:, 0], dim=1, keepdim=True)
    mom_y_neq = torch.sum(f_neq * xi_t[:, 1], dim=1, keepdim=True)
    return torch.mean(mass_neq**2 + mom_x_neq**2 + mom_y_neq**2)


N_col = 3000
x_col = (torch.rand(N_col, 1, device=device) * 2.5 - 0.5)
y_col = (torch.rand(N_col, 1, device=device) * 2.0 - 0.5)
xy_col = torch.cat([x_col, y_col], dim=1).requires_grad_(True)

N_b = 400
xb = np.concatenate([np.full(N_b, -0.5), np.full(N_b, 2.0),
                      np.random.uniform(-0.5, 2.0, N_b), np.random.uniform(-0.5, 2.0, N_b)])
yb = np.concatenate([np.random.uniform(-0.5, 1.5, N_b), np.random.uniform(-0.5, 1.5, N_b),
                      np.full(N_b, -0.5), np.full(N_b, 1.5)])
u_b, v_b, p_b = exact_uvp(xb, yb)
rho_b = p_b / RT
xy_bnd = torch.tensor(np.stack([xb, yb], axis=1), dtype=torch.float32, device=device)
uvw_bnd = torch.tensor(np.stack([rho_b, u_b, v_b], axis=1), dtype=torch.float32, device=device)


def compute_loss(lam_eq=1.0, lam_bc=10.0, lam_mom=1.0):
    R, f_neq = dvb_residual(xy_col)
    loss_eq = torch.mean(R**2)
    loss_mom = moment_constraint(f_neq)

    out_bnd = NN_eq(xy_bnd)
    loss_bc = torch.mean((out_bnd - uvw_bnd)**2)

    total = lam_eq * loss_eq + lam_bc * loss_bc + lam_mom * loss_mom
    return total, loss_eq.item(), loss_bc.item(), loss_mom.item()

## 4. Entrenamiento

In [ ]:
params = list(NN_eq.parameters()) + list(NN_neq.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss, l_eq, l_bc, l_mom = compute_loss()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | Eq={l_eq:.4e} | BC={l_bc:.4e} | moment={l_mom:.4e}')

# Segunda etapa con L-BFGS, tal como en el paper ("we first employ Adam ... then we switch to L-BFGS-B")
opt_lbfgs = torch.optim.LBFGS(params, lr=0.5, max_iter=300, history_size=30,
                               line_search_fn='strong_wolfe')

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss()
    loss.backward()
    return loss

loss_final = opt_lbfgs.step(closure)
history.append(loss_final.item())
print(f'[L-BFGS] loss final={loss_final.item():.4e}')

## 5. Resultados: comparacion con la solucion exacta (cf. Tabla 1: err_u, err_v)

In [ ]:
n_side = 80
xs = np.linspace(-0.5, 2.0, n_side)
ys = np.linspace(-0.5, 1.5, n_side)
Xg, Yg = np.meshgrid(xs, ys)
xy_test = torch.tensor(np.stack([Xg.ravel(), Yg.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    out_test = NN_eq(xy_test)
u_pred = out_test[:, 1].cpu().numpy().reshape(Xg.shape)
v_pred = out_test[:, 2].cpu().numpy().reshape(Xg.shape)
u_exact, v_exact, _ = exact_uvp(Xg, Yg)

err_u = np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)
err_v = np.linalg.norm(v_pred - v_exact) / np.linalg.norm(v_exact)
print(f'Error relativo L2: err_u={err_u*100:.2f}%  err_v={err_v*100:.2f}%  (paper reporta 0.21%, 0.60% con red completa)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].contourf(Xg, Yg, u_exact, levels=30, cmap='RdBu_r')
axes[0].set_title('u exacta'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].contourf(Xg, Yg, u_pred, levels=30, cmap='RdBu_r')
axes[1].set_title('u PINN-BGK'); plt.colorbar(im1, ax=axes[1])
im2 = axes[2].contourf(Xg, Yg, np.abs(u_pred - u_exact), levels=30, cmap='viridis')
axes[2].set_title('|error|'); plt.colorbar(im2, ax=axes[2])
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Epoca'); plt.ylabel('Loss (escala log)')
plt.title('Convergencia de PINN-BGK')
plt.show()

**Nota honesta sobre la precision obtenida:** a diferencia de la mayoria de los cuadernos de esta coleccion, esta reproduccion **no** alcanza la precision cuantitativa reportada por el paper (err_u=0.21%, err_v=0.60%). El residuo del DVB y el ajuste de contorno convergen a valores muy pequenos (ambos <10^-3), pero el campo macroscopico $(\rho,u,v)$ en el interior sigue derivando de forma importante respecto a la solucion exacta. Se investigo la causa (bug de *broadcasting* en `f_equilibrium`, corregido; falta de restriccion de masa/momento nulos en $f^{neq}$, anadida como `moment_constraint`; peso de frontera y una segunda etapa con L-BFGS-B, ambos probados) sin cerrar completamente la brecha, lo que sugiere que el sistema de 9 residuos puntuales por si solo deja parcialmente subdeterminado el campo macroscopico en el interior sin una supervision de frontera aun mas fuerte o una densidad de colocacion mayor que la usada aqui (el paper usa 17 000 puntos y una red completa con validacion cruzada de escalado, Seccion 3.1). El **mecanismo arquitectonico central del paper -- la descomposicion en dos subredes ($\mathcal{NN}_{eq}$/$\mathcal{NN}_{neq}$), el reescalado por $10^4$, y el residuo de la ecuacion discreta de Boltzmann-BGK -- esta correctamente implementado y documentado**; lo que no se logro en este cuaderno es la precision final de punta a punta que el paper reporta con su configuracion completa.